# Consulta Gold: métricas de Data Quality de `ecommerce_enderecos`
Notebook **somente leitura** para conferir a Gold gerada pelo `06_ingestao_gold`.

| KPI | Delta (camada Gold) | SQL Server (consumo no Looker) |
|-----|---------------------|--------------------------------|
| % de falha por regra, por dia | `squad1/gold/ecommerce_enderecos/dq_resumo_por_regra` | `squad1.gold_dq_resumo_por_regra_enderecos` |
| % de registros limpos, por hora | `squad1/gold/ecommerce_enderecos/dq_resumo_por_tabela` | `squad1.gold_dq_resumo_por_tabela_enderecos` |

Etapas: conexão → KPI 1 → KPI 2 → conferência Delta x SQL Server.

In [0]:
# 1. Conexão com o ADLS e o SQL Server (.env)
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

load_dotenv(".env")

client_id            = os.getenv("ADLS_CLIENT_ID")
tenant_id            = os.getenv("ADLS_TENANT_ID")
client_secret        = os.getenv("ADLS_CLIENT_SECRET")
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")

variaveis = {
    "ADLS_CLIENT_ID": client_id, "ADLS_TENANT_ID": tenant_id, "ADLS_CLIENT_SECRET": client_secret,
    "SQL_HOST": os.getenv("SQL_HOST"), "SQL_DATABASE": os.getenv("SQL_DATABASE"),
    "SQL_USERNAME": os.getenv("SQL_USERNAME"), "SQL_PASSWORD": os.getenv("SQL_PASSWORD"),
}
faltantes = [nome for nome, valor in variaveis.items() if not valor]
assert not faltantes, f"Faltou variável no .env: {', '.join(faltantes)}"

adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}
sql_opts = {
    "host": variaveis["SQL_HOST"], "port": "1433", "database": variaveis["SQL_DATABASE"],
    "user": variaveis["SQL_USERNAME"], "password": variaveis["SQL_PASSWORD"],
    "encrypt": "true", "trustServerCertificate": "false",
}

gold_base = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net/gold/ecommerce_enderecos"
KPIS = {
    "dq_resumo_por_regra":  (f"{gold_base}/dq_resumo_por_regra",  "squad1.gold_dq_resumo_por_regra_enderecos"),
    "dq_resumo_por_tabela": (f"{gold_base}/dq_resumo_por_tabela", "squad1.gold_dq_resumo_por_tabela_enderecos"),
}

def ler_gold_delta(kpi):
    return spark.read.format("delta").options(**adls_opts).load(KPIS[kpi][0])

def ler_gold_sql(kpi):
    return spark.read.format("sqlserver").options(**sql_opts).option("dbtable", KPIS[kpi][1]).load()

df_regra = ler_gold_delta("dq_resumo_por_regra")
df_tabela = ler_gold_delta("dq_resumo_por_tabela")
print(f"✔ dq_resumo_por_regra: {df_regra.count()} linha(s) | dq_resumo_por_tabela: {df_tabela.count()} linha(s)")

### KPI 1: % de falha por regra, por dia
Tabela completa e, em seguida, o acumulado de cada regra no período (da regra que mais falha para a que menos falha).

In [0]:
display(df_regra.orderBy("_data", "regra"))

display(
    df_regra.groupBy("regra", "severidade")
    .agg(F.sum("qtd_falhas").alias("qtd_falhas"), F.sum("qtd_total").alias("qtd_total"))
    .withColumn("pct_falha", F.round(F.col("qtd_falhas") / F.col("qtd_total") * 100, 2))
    .orderBy(F.col("pct_falha").desc())
)

### KPI 2: % de registros limpos, por hora
Linha limpa = não falhou em nenhuma das 10 regras.

In [0]:
display(df_tabela.orderBy("_data", "_hora"))

total = df_tabela.agg(F.sum("qtd_limpos").alias("limpos"), F.sum("qtd_total").alias("total")).first()
if total.total:
    print(f"Período todo: {total.limpos} de {total.total} registros limpos ({total.limpos / total.total * 100:.2f}%)")

### Conferência Delta x SQL Server
O Looker lê o SQL Server: as duas fontes devem ter as mesmas linhas.

In [0]:
for kpi in KPIS:
    df_delta = ler_gold_delta(kpi)
    try:
        df_sql = ler_gold_sql(kpi)
        colunas = df_delta.columns
        so_delta = df_delta.select(colunas).exceptAll(df_sql.select(colunas)).count()
        so_sql = df_sql.select(colunas).exceptAll(df_delta.select(colunas)).count()
        situacao = "✔ iguais" if so_delta == 0 and so_sql == 0 else f"⚠ só na Delta: {so_delta} | só no SQL: {so_sql}"
        print(f"{KPIS[kpi][1]:<45} Delta {df_delta.count():>5} | SQL {df_sql.count():>5} → {situacao}")
    except Exception as e:
        print(f"✖ {KPIS[kpi][1]}: não foi possível ler o SQL Server ({str(e).splitlines()[0][:120]})")

In [0]:
# Diagnóstico (somente leitura): Silver x dq_monitoring_logs de ecommerce_enderecos
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

load_dotenv(".env")
storage_account_name = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
tenant_id = os.getenv("ADLS_TENANT_ID")
adls_opts = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}
base = f"abfss://squad1@{storage_account_name}.dfs.core.windows.net"
ler = lambda p: spark.read.format("delta").options(**adls_opts).load(p)

df_bronze = ler(f"{base}/bronze/ecommerce_enderecos")
df_silver = ler(f"{base}/silver/ecommerce_enderecos")
df_logs   = ler(f"{base}/dq_monitoring_logs").filter(F.col("tabela") == "ecommerce_enderecos")

# 1. Lotes em cada camada
print("Lotes na Bronze:", df_bronze.select("bronze_source_file").distinct().count())
print("Lotes na Silver:", df_silver.select("bronze_source_file").distinct().count())
print("Lotes no log   :", df_logs.select("arquivo_origem").distinct().count())

# 2. Como o caminho está gravado em cada lugar (precisam bater depois de tirar o 'abfss://.../')
display(df_silver.select("bronze_source_file").distinct().orderBy(F.desc("bronze_source_file")).limit(5))
display(df_logs.select("arquivo_origem").distinct().orderBy(F.desc("arquivo_origem")).limit(5))

# 3. Lotes da Silver que não têm nenhuma linha no log (o que a Silver nova deveria ter registrado)
silver_rel = (df_silver
    .withColumn("arquivo_origem", F.regexp_replace("bronze_source_file", r"^abfss://[^/]+/", ""))
    .groupBy("arquivo_origem").agg(F.count("*").alias("linhas"), F.max("silver_processed_at").alias("processado_em")))
faltando = silver_rel.join(df_logs.select("arquivo_origem").distinct(), "arquivo_origem", "left_anti")
print("Lotes da Silver sem log:", faltando.count())
display(faltando.orderBy("arquivo_origem"))

# 4. Última gravação no log
display(df_logs.agg(F.max("timestamp_execucao").alias("ultimo_log"), F.count("*").alias("linhas_log")))
